# 30분 첫 모델

작은 거시 모형 하나를 처음부터 끝까지 만듭니다: 자료 열기 → 방정식 추정 → 연립 모델 → 과거 재현 → 시나리오 → 전망 → 저장.
모든 코드는 그대로 복사해 실행할 수 있고, 아래에 보이는 출력은 실제로 실행한 결과입니다(문서 검사가 매번 다시 돌려 확인합니다).
같은 내용을 스크립트([examples/start/first-model.py](../start/first-model.py))와 노트북
([examples/notebooks/first-model.ipynb](first-model.ipynb))으로도 받을 수 있습니다.
이 문서는 짧은 길잡이이고, 기능별 자세한 설명은 [튜토리얼](../../docs/tutorial/README.md)에 있습니다.

준비: [설치](../../docs/start/install.md)를 마쳤다면 Python(또는 Jupyter)에서 시작합니다.

## 1. 예제 자료 열기

Equion에는 문서가 함께 쓰는 작은 가상 자료가 들어 있습니다. `toy_q`는 소비·투자·정부지출·국민소득·물가가 든 분기 자료이고,
알려진 '참 모형'에서 만들었기 때문에 추정 결과를 참값과 견줄 수 있습니다.

In [ ]:
import equion as eqn

wf = eqn.datasets.load("toy_q")          # 워크파일: 시리즈·표본·객체를 담는 작업 단위
print(wf)

어떤 시리즈가 있는지는 메타데이터 표로 봅니다.

In [ ]:
eqn.show(wf.metadata())

참 모형은 이렇습니다(투자 `I`는 지난 분기 소득과 금리에, 소비 `CN`은 이번 분기 소득과 지난 분기 소비에 반응하고,
국민소득 `Y`는 항등식으로 정해집니다).

In [ ]:
print(eqn.datasets.describe("toy_q"))

## 2. 방정식 추정

방정식은 글자로 씁니다. 첫 항이 종속변수, `c`는 상수항, `Y(-1)`은 한 분기 전 값입니다.

In [ ]:
eq_i = wf.equation("eq_i").ls("I c Y(-1) R")      # 최소제곱(OLS)
eqn.show(eq_i)

`R`의 계수 −0.96이 참값 −1.0에 가깝습니다. 결과는 pandas 객체로 바로 꺼낼 수 있습니다.

In [ ]:
print(eq_i.coefs.round(3).to_dict())
print(round(eq_i.stats["r_squared"], 4))

소비식에서는 이번 분기 소득 `Y`가 소비 자신에 의해 정해지는 **내생변수**라서 최소제곱이 치우칩니다. 도구변수를 쓰는
2단계 최소제곱(`tsls`)과 견줘 봅니다(참값은 `Y` 0.5, `CN(-1)` 0.3).

In [ ]:
ols = wf.equation("eq_cn_ols").ls("CN c Y CN(-1)")
eq_cn = wf.equation("eq_cn").tsls("CN c Y CN(-1) @inst G R CN(-1) Y(-1)")
print("OLS ", ols.coefs.round(3).to_dict())
print("TSLS", eq_cn.coefs.round(3).to_dict())

물가식은 차분 꼴입니다. `dlog(P)`는 `log(P) − log(P(-1))`, 곧 분기 물가 상승률입니다.

In [ ]:
eq_p = wf.equation("eq_p").ls("dlog(P) c dlog(Y) dlog(POIL)")
print(eq_p.coefs.round(4).to_dict())

## 3. 연립 모델 만들기

추정한 식들과 항등식을 모델 하나로 묶습니다. 내생변수는 각 식의 종속변수(`I`·`CN`·`P`)와 항등식의 좌변(`Y`)입니다.

In [ ]:
m = wf.model("toy").merge(eq_i, eq_cn, eq_p).append("Y = CN + I + G")
print(m.endogenous, m.exogenous)
eqn.show(m.blocks())                      # 함께 풀어야 하는 변수 묶음(동시 블록)과 풀이 순서

## 4. 과거 재현(Historical Simulation)

추정한 모델이 과거를 얼마나 따라가는지, 지난 기의 풀이 값을 다음 기 시차로 쓰는 **동적 풀이**로 봅니다.
결과는 별칭을 붙인 시리즈(`Y_0`·`CN_0` …)로 워크파일에 들어갑니다.

In [ ]:
wf.smpl("1975Q1", "2019Q4")              # 풀이·추정·그래프의 기본 구간(표본)
base = m.solve()
eqn.show(base.summary())

In [ ]:
print(wf.to_pandas(["Y", "Y_0"], labels=True).loc["2019Q1":"2019Q4"].round(1))

## 5. 시나리오

정부지출을 2015년부터 10% 늘리면 소득과 소비가 얼마나 달라질까요? 시나리오는 기준선(`_0`)과 다른 별칭(`_1`)으로 결과를 남깁니다.

In [ ]:
sc = m.scenario("g_up")
sc.shock("G", 10, "2015Q1", "2019Q4", kind="pct")    # G를 10% 높인다
m.solve("2015Q1", "2019Q4")
cmp = wf.to_pandas(["Y_0", "Y_1"], labels=True).loc["2015Q1":"2015Q4"]
cmp["차이(%)"] = (cmp["Y_1"] / cmp["Y_0"] - 1) * 100
print(cmp.round(2))

정부지출이 늘면 소득이 늘고(승수), 소득이 늘면 소비와 다음 분기 투자가 다시 늘어 효과가 커집니다.
어느 경로로 차이가 생겼는지는 [시나리오 튜토리얼](../../docs/tutorial/05-scenario.md)의 기여도 분해로 나눠 봅니다.

## 6. 전망

자료는 2019Q4에서 끝납니다. 2020~2021년을 전망하려면 외생변수(`G`·`R`·`POIL`)의 앞날 경로가 필요합니다.
여기서는 최근 추세를 이어 채웁니다(`exog_fill="trend"`).

In [ ]:
m.scenario("baseline")                    # 기준선 시나리오로 돌아간다
fc = m.forecast("2020Q1", "2021Q4", exog_fill="trend")
print(wf.to_pandas(["Y_0", "CN_0", "I_0"], labels=True).loc["2020Q1":"2021Q4"].round(1))

## 7. 그래프와 저장

In [ ]:
g = wf.graph("g_y", "Y Y_0 Y_1", start="2010Q1", end="2021Q4", title="국민소득: 실측·기준선·시나리오")
g.save("y.html")                          # html(plotly 없이도 됨) — plotly.js는 인터넷에서 부른다. 인터넷 없이: g.output().save("y.html", include_js="inline")
wf.save("toy")                            # toy.eqw/ — 자료·방정식 계수·모델·시나리오를 모두 저장
wf2 = eqn.wfopen("toy.eqw")               # 다시 열면 재추정 없이 그대로
print(wf2.obj("eq_i").coefs.round(3).to_dict())

## 다음 단계

- 각 단계를 자세히: [튜토리얼 1~9](../../docs/tutorial/README.md)
- 할 일별 짧은 처방: [레시피](../../docs/howto/README.md)
- 방정식 문법 전체: [방정식 문법](../../docs/explain/equations.md)
- 모든 함수: [API 참조](../../docs/reference/api.md) · 한 장 요약: [치트시트](../../docs/start/cheatsheet.md)